In [ ]:
!pip install -q xgboost scikit-learn pandas numpy pyarrow

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Drive
drive.mount('/content/drive')

# Path to your data folder in Drive
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/audit_anomaly_data')

# Verify the files are accessible
print("Files in Drive folder:")
for f in DRIVE_DATA_DIR.iterdir():
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name} ({size_mb:.1f} MB)")

Mounted at /content/drive
Files in Drive folder:
  train_transaction.csv (651.7 MB)
  train_identity.csv (25.3 MB)


In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

from google.colab import files

In [ ]:
# =============================================================
print("=== STEP 1: LOAD AND MERGE CSVs ===")
# =============================================================
drive_dir = Path("/content/drive/MyDrive/audit_anomaly_data")
tx_path = drive_dir / "train_transaction.csv"
id_path = drive_dir / "train_identity.csv"

train_tx = pd.read_csv(tx_path)
train_id = pd.read_csv(id_path)
print(f"  train_transaction: {train_tx.shape}")
print(f"  train_identity:    {train_id.shape}")

df = train_tx.merge(train_id, on="TransactionID", how="left")
print(f"  merged:            {df.shape}")
print(f"  overall fraud rate: {df['isFraud'].mean():.4f}")

=== STEP 1: LOAD AND MERGE CSVs ===
  train_transaction: (590540, 394)
  train_identity:    (144233, 41)
  merged:            (590540, 434)
  overall fraud rate: 0.0350


In [ ]:
# =============================================================
print("\n=== STEP 2: STRATIFIED SAMPLE TO 50,000 ROWS ===")
# =============================================================
df_sample, _ = train_test_split(
    df,
    train_size=50_000,
    stratify=df["isFraud"],
    random_state=42,
)
df_sample = df_sample.reset_index(drop=True)
print(f"  sample shape: {df_sample.shape}")
print(f"  sample fraud rate: {df_sample['isFraud'].mean():.4f}")


=== STEP 2: STRATIFIED SAMPLE TO 50,000 ROWS ===
  sample shape: (50000, 434)
  sample fraud rate: 0.0350


In [ ]:
# =============================================================
print("\n=== STEP 3: ENGINEER TransactionHour FEATURE ===")
# =============================================================
df_sample["TransactionHour"] = (df_sample["TransactionDT"] % 86400) / 3600
print(df_sample["TransactionHour"].describe().round(2))


=== STEP 3: ENGINEER TransactionHour FEATURE ===
count    50000.00
mean        14.36
std          7.61
min          0.00
25%          6.82
50%         16.85
75%         20.39
max         24.00
Name: TransactionHour, dtype: float64


In [ ]:
# =============================================================
print("\n=== STEP 4: SELECT FEATURE LIST ===")
# =============================================================
feature_list = [
    "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "dist1",
    "P_emaildomain", "R_emaildomain",
    "C1", "C2", "C13",
    "V12", "V13", "V14",
    "TransactionHour",
]
categorical_features = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain"]
numerical_features = [f for f in feature_list if f not in categorical_features]

# Keep TransactionID alongside for later (we'll join back for the scored output).
X = df_sample[feature_list].copy()
y = df_sample["isFraud"].copy()
tx_ids = df_sample["TransactionID"].copy()

# Snapshot of pre-encoding originals — used later for the display DataFrame.
display_lookup = df_sample.set_index("TransactionID")[
    ["TransactionAmt", "ProductCD", "card4", "card6", "P_emaildomain", "TransactionHour"]
]

print(f"  features: {len(feature_list)} "
      f"(numerical={len(numerical_features)}, categorical={len(categorical_features)})")


=== STEP 4: SELECT FEATURE LIST ===
  features: 20 (numerical=15, categorical=5)


In [ ]:

# =============================================================
print("\n=== STEP 5: HANDLE MISSING DATA ===")
# =============================================================
# Numerical → median fill (medians captured for later scoring of new data)
numerical_medians = {}
for col in numerical_features:
    med = X[col].median()
    numerical_medians[col] = med
    X[col] = X[col].fillna(med)

# Categorical → fill 'unknown' then label-encode
label_encoders = {}
for col in categorical_features:
    X[col] = X[col].fillna("unknown").astype(str)
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

print(f"  remaining nulls: {int(X.isnull().sum().sum())}")





=== STEP 5: HANDLE MISSING DATA ===
  remaining nulls: 0


In [ ]:
# =============================================================
print("\n=== STEP 6: TRAIN/TEST SPLIT (80/20, stratified) ===")
# =============================================================
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, tx_ids,
    test_size=0.20,
    stratify=y,
    random_state=42,
)
print(f"  train: {X_train.shape}, fraud rate: {y_train.mean():.4f}")
print(f"  test : {X_test.shape}, fraud rate: {y_test.mean():.4f}")




=== STEP 6: TRAIN/TEST SPLIT (80/20, stratified) ===
  train: (40000, 20), fraud rate: 0.0350
  test : (10000, 20), fraud rate: 0.0350


In [ ]:
# =============================================================
print("\n=== STEP 7: TRAIN MODELS ===")
# =============================================================
# --- IsolationForest ---
print("  [1/3] IsolationForest...")
iso = IsolationForest(contamination=0.035, random_state=42, n_jobs=-1)
iso.fit(X_train)

# --- XGBoost ---
print("  [2/3] XGBoost...")
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / max(pos, 1)
print(f"        scale_pos_weight = {scale_pos_weight:.2f}  (neg={neg}, pos={pos})")
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    tree_method="hist",
)
xgb_model.fit(X_train, y_train)

# --- LogisticRegression (with StandardScaler) ---
print("  [3/3] LogisticRegression...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    n_jobs=-1,
)
lr_model.fit(X_train_scaled, y_train)



=== STEP 7: TRAIN MODELS ===
  [1/3] IsolationForest...
  [2/3] XGBoost...
        scale_pos_weight = 27.57  (neg=38600, pos=1400)
  [3/3] LogisticRegression...


LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1,
                   random_state=42)

In [ ]:
# =============================================================
print("\n=== STEP 8: GENERATE & ENSEMBLE ANOMALY SCORES ===")
# =============================================================
# IsolationForest.decision_function: higher = more normal. Negate so higher = more anomalous.
iso_raw = -iso.decision_function(X_test)
xgb_raw = xgb_model.predict_proba(X_test)[:, 1]
lr_raw = lr_model.predict_proba(X_test_scaled)[:, 1]

# Normalize each independently to [0, 1] so they're on a comparable scale before averaging.
iso_score = MinMaxScaler().fit_transform(iso_raw.reshape(-1, 1)).ravel()
xgb_score = MinMaxScaler().fit_transform(xgb_raw.reshape(-1, 1)).ravel()
lr_score = MinMaxScaler().fit_transform(lr_raw.reshape(-1, 1)).ravel()

ensemble_score = (iso_score + xgb_score + lr_score) / 3.0
print(f"  ensemble score range: [{ensemble_score.min():.4f}, {ensemble_score.max():.4f}]")
print(f"  mean ensemble score:  {ensemble_score.mean():.4f}")



=== STEP 8: GENERATE & ENSEMBLE ANOMALY SCORES ===
  ensemble score range: [0.1001, 0.9063]
  mean ensemble score:  0.2746


In [ ]:
# =============================================================
print("\n=== STEP 9: EVALUATE ENSEMBLE (threshold = 0.5) ===")
# =============================================================
y_pred = (ensemble_score > 0.5).astype(int)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, ensemble_score)

print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1:        {f1:.4f}")
print(f"  AUC:       {auc:.4f}")
print(f"  Predicted positives: {int(y_pred.sum())} / {len(y_pred)}")



=== STEP 9: EVALUATE ENSEMBLE (threshold = 0.5) ===
  Precision: 0.3136
  Recall:    0.5000
  F1:        0.3855
  AUC:       0.8229
  Predicted positives: 558 / 10000


In [ ]:
# =============================================================
print("\n=== STEP 10: SAVE MODELS BUNDLE (models_bundle.pkl) ===")
# =============================================================
bundle = {
    "isolation_forest": iso,
    "xgboost": xgb_model,
    "logistic_regression": lr_model,
    "scaler": scaler,
    "label_encoders": label_encoders,
    "feature_list": feature_list,
    "numerical_medians": numerical_medians,
}
bundle_path = Path("models_bundle.pkl")
with open(bundle_path, "wb") as f:
    pickle.dump(bundle, f)
print(f"  wrote {bundle_path}  ({bundle_path.stat().st_size / 1024:.1f} KB)")


=== STEP 10: SAVE MODELS BUNDLE (models_bundle.pkl) ===
  wrote models_bundle.pkl  (1751.4 KB)


In [ ]:

# =============================================================
print("\n=== STEP 11: BUILD TOP-1000 SCORED TRANSACTIONS DATAFRAME ===")
# =============================================================
# Pull pre-encoded display values from the lookup so categoricals are readable strings.
disp = display_lookup.loc[ids_test.values].reset_index(drop=True)

results = pd.DataFrame({
    "TransactionID": ids_test.values,
    "TransactionAmt": disp["TransactionAmt"].values,
    "ProductCD": disp["ProductCD"].values,
    "card4": disp["card4"].values,
    "card6": disp["card6"].values,
    "P_emaildomain": disp["P_emaildomain"].values,
    "TransactionHour": disp["TransactionHour"].values,
    "isFraud": y_test.values,
    "iso_score": iso_score,
    "xgb_score": xgb_score,
    "lr_score": lr_score,
    "ensemble_score": ensemble_score,
})

top1000 = results.nlargest(1000, "ensemble_score").reset_index(drop=True)
print(f"  top-1000 shape: {top1000.shape}")
print(f"  true frauds in top-1000: {int(top1000['isFraud'].sum())} "
      f"(precision@1000 = {top1000['isFraud'].mean():.4f})")



=== STEP 11: BUILD TOP-1000 SCORED TRANSACTIONS DATAFRAME ===
  top-1000 shape: (1000, 12)
  true frauds in top-1000: 206 (precision@1000 = 0.2060)


In [ ]:
# =============================================================
print("\n=== STEP 12: SAVE PARQUET (scored_transactions.parquet) ===")
# =============================================================
parquet_path = Path("scored_transactions.parquet")
top1000.to_parquet(parquet_path, index=False)
print(f"  wrote {parquet_path}  ({parquet_path.stat().st_size / 1024:.1f} KB)")


=== STEP 12: SAVE PARQUET (scored_transactions.parquet) ===
  wrote scored_transactions.parquet  (59.1 KB)


In [ ]:
# =============================================================
print("\n=== STEP 13: TRIGGER DOWNLOADS ===")
# =============================================================
files.download(str(bundle_path))
files.download(str(parquet_path))
print("  download prompts triggered for both files.")
print("\nDone.")


=== STEP 13: TRIGGER DOWNLOADS ===


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  download prompts triggered for both files.

Done.
